In [1]:
!pip install selenium==4.6
!pip install yt-dlp


[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from selenium import webdriver
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.common.by import By
import requests
import os
import csv
import time
import hashlib

# Ruta donde descomprimiste geckodriver.exe
gecko_path = "C:/Program Files/WebDrivers/geckodriver.exe"

options = webdriver.FirefoxOptions()
options.add_argument("--start-maximized")

driver = webdriver.Firefox(service=Service(gecko_path), options=options)

driver.get("https://www.facebook.com")
# driver.get("https://x.com")
time.sleep(10)

input("Inicia sesión manualmente y presiona ENTER aquí...")

''

In [15]:
hashtag = "deslizamento de Terra"
location_hashtag = "Ipatinga brasil 2025"
class_label = "deslizamento de terra"
max_images = 30
# inundações

# --- URL de búsqueda en Facebook ---
search_url = f"https://www.facebook.com/search/top?q={hashtag}%20{location_hashtag}"

# --- URL de búsqueda en X ---
# search_url = f"https://x.com/search?q={hashtag}%20{location_hashtag}"

# --- Inicializar navegador ---
driver.get(search_url)
time.sleep(10)

# --- Preparar carpetas y rutas ---
folder_name = class_label.lower().replace(" ", "_")
img_base_path = os.path.join("dataset", folder_name)
os.makedirs(img_base_path, exist_ok=True)
csv_path = os.path.join("dataset", "dataset.csv")


# --- Leer imágenes ya descargadas ---
descargados = set()
if os.path.exists(csv_path) and os.path.getsize(csv_path) > 0:
    with open(csv_path, mode='r', encoding='utf-8', errors='ignore') as csv_file:
        reader = csv.DictReader(csv_file)
        for row in reader:
            descargados.add(row["img_path"])

# --- Abrir CSV para escritura ---
csv_existe = os.path.exists(csv_path)
csv_file = open(csv_path, mode='a', newline='', encoding='utf-8')
csv_writer = csv.writer(csv_file)
if not csv_existe:
    csv_writer.writerow(["class_label", "img_path", "post_text"])

# --- Scroll para cargar más contenido ---
for _ in range(5):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(5)

ver_mais_buttons = driver.find_elements(By.XPATH, '//div[contains(text(), "Ver más")]')
for btn in ver_mais_buttons:
    try:
        driver.execute_script("arguments[0].click();", btn)
        time.sleep(0.3)
    except Exception as e:
        print(f"[!] No se pudo hacer clic en 'Ver más': {e}")

# --- Extraer posts e imágenes ---
# Facebook
posts = driver.find_elements(By.XPATH, '//div[@data-ad-preview="message"]')
imagenes = driver.find_elements(By.XPATH, '//img[contains(@src, "scontent")]')

# X
# posts =  driver.find_elements(By.XPATH, '//div[@data-testid="tweetText"]')
# imagenes = driver.find_elements(By.XPATH, '//img[contains(@src, "twimg.com/media")]')

# Instagram


# --- Descargar imágenes con texto ---
guardadas = 0
for i in range(len(posts)):
    if guardadas >= max_images:
        break

    texto = posts[i].text.strip()
    img_url = imagenes[i].get_attribute("src") if i < len(imagenes) else None

    if texto and img_url:
        img_hash = hashlib.md5(img_url.encode()).hexdigest()
        img_path = os.path.join(img_base_path, f"{img_hash}.jpg")

        if img_path not in descargados:
            try:
                # img_data = requests.get(img_url).content
                headers = {"User-Agent": "Mozilla/5.0"}
                img_data = requests.get(img_url, headers=headers).content
                with open(img_path, 'wb') as handler:
                    handler.write(img_data)

                texto_limpio = texto.replace("\n", " ")
                csv_writer.writerow([class_label, img_path, texto_limpio])
                descargados.add(img_path)
                guardadas += 1
                print(f"[{guardadas}/{max_images}] Imagen guardada: {img_path}")
            except Exception as e:
                print(f"[X] Error al descargar imagen: {e}")

# --- Cierre ---
csv_file.close()
# driver.quit()
print("✅ Dataset generado en:", img_base_path)


[1/30] Imagen guardada: dataset\deslizamento_de_terra\8d9534651ada418e23c2a58c4635eb09.jpg
[2/30] Imagen guardada: dataset\deslizamento_de_terra\60ebc1ec45614d70dfa0f081922627e2.jpg
[3/30] Imagen guardada: dataset\deslizamento_de_terra\da4dbea200d2a5132ce80d9cd7d48497.jpg
[4/30] Imagen guardada: dataset\deslizamento_de_terra\e4bb7a1c5406a0ffbc40b61714eada2c.jpg
[5/30] Imagen guardada: dataset\deslizamento_de_terra\044123002d8893559d786500ead6e936.jpg
[6/30] Imagen guardada: dataset\deslizamento_de_terra\cff1b22f631ab24eecd9a35a0d8deebf.jpg
[7/30] Imagen guardada: dataset\deslizamento_de_terra\eb8dcbaacc014514c792647c0a65f747.jpg
[8/30] Imagen guardada: dataset\deslizamento_de_terra\02bc89afdf358a91f88573071520e935.jpg
[9/30] Imagen guardada: dataset\deslizamento_de_terra\d1abe2102163bb1e598127194cbc94e9.jpg
[10/30] Imagen guardada: dataset\deslizamento_de_terra\b4790bbac1ab373bddf234b400f29d3b.jpg
[11/30] Imagen guardada: dataset\deslizamento_de_terra\5130bb9856e2e76d714a9f1d03288444.j

In [13]:
import os
import csv

img_base_path = os.path.join("dataset", "deslizamento_de_terra")

csv_path = 'dataset\dataset.csv'

# --- Leer CSV y filtrar registros válidos ---
temp_rows = []
with open(csv_path, mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    for row in reader:
        img_path = row["img_path"]
        if os.path.exists(img_path):  # Verifica si el archivo existe
            temp_rows.append(row)
        else:
            print(f"[!] Imagen no encontrada y será eliminada del CSV: {img_path}")

# --- Reescribir CSV sin las filas eliminadas ---
with open(csv_path, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=["class_label", "img_path", "post_text"])
    writer.writeheader()
    writer.writerows(temp_rows)

print("✅ Revisión completa. CSV actualizado.")


[!] Imagen no encontrada y será eliminada del CSV: dataset\deslizamento_de_terra\fc8623f94be811cc46ff0b9d157981fb.jpg
[!] Imagen no encontrada y será eliminada del CSV: dataset\deslizamento_de_terra\a853cce6a24e529546777c9feca2aad2.jpg
[!] Imagen no encontrada y será eliminada del CSV: dataset\deslizamento_de_terra\4462fb50e620c633e50b8c7cf95407d9.jpg
[!] Imagen no encontrada y será eliminada del CSV: dataset\deslizamento_de_terra\b896d7495cef3a3038bfd004640964a4.jpg
[!] Imagen no encontrada y será eliminada del CSV: dataset\deslizamento_de_terra\6a996c3b3dc64db98b606492cf50af39.jpg
[!] Imagen no encontrada y será eliminada del CSV: dataset\deslizamento_de_terra\1e6dbd187efcc7a0f12bc3e975070b03.jpg
[!] Imagen no encontrada y será eliminada del CSV: dataset\deslizamento_de_terra\fa5872ab84ec9fd2d8f35bd47a9c07e2.jpg
[!] Imagen no encontrada y será eliminada del CSV: dataset\deslizamento_de_terra\b19f325eabe79a71bb9e5f67dae83a41.jpg
[!] Imagen no encontrada y será eliminada del CSV: datas

In [ ]:
import os
import csv
from PIL import Image
import matplotlib.pyplot as plt

csv_path = 'dataset\dataset.csv'  # <-- cambia esta ruta
img_base_path = os.path.join("dataset", "deslizamento_de_terra")
temp_rows = []

# Leer CSV original
with open(csv_path, mode='r', encoding='utf-8') as file:

    reader = csv.DictReader(file)
    for row in reader:

        img_path = row["img_path"]
        texto = row["post_text"]

        if not img_path.startswith(img_base_path):
            continue 

        if not os.path.exists(img_path):
            print(f"[!] Imagen no encontrada: {img_path}")
            continue
  
        # Mostrar imagen
        with Image.open(img_path) as img:
            plt.imshow(img)
            plt.axis('off')
            plt.title("¿Deseas eliminar esta imagen?")
            plt.show()

        # Mostrar texto
        print(f"\nTexto:\n{texto}\n")

        # Preguntar si desea conservarla
        opcion = input("¿Eliminar esta imagen? (s = sí / enter = no): ").strip().lower()
        if opcion == 's':
            os.remove(img_path)
            print(f"[✓] Imagen eliminada: {img_path}\n")
        else:
            temp_rows.append(row)  # conservar esta fila

# # Reescribir CSV sin las filas eliminadas
# with open(csv_path, mode='w', newline='', encoding='utf-8') as file:
#     writer = csv.DictWriter(file, fieldnames=["class_label", "img_path", "post_text"])
#     writer.writeheader()
#     writer.writerows(temp_rows)

# print("✅ Revisión completa. CSV actualizado.")


In [ ]:



# hashtag = "inundações ruas"
# location_hashtag = "Brasil"
# search_url = f"https://www.facebook.com/search/top?q={hashtag}%20{location_hashtag}"


# class_label = "chuva_forte"
# max_images = 30

# driver.get(search_url)
# time.sleep(10)

# # Crear carpetas
# folder_name = hashtag.lower().replace(" ", "_")
# base_path = os.path.join("dataset", folder_name)
# img_base_path = os.path.join(base_path, "img")
# os.makedirs(img_base_path, exist_ok=True)

# # Crear CSV
# csv_path = os.path.join(base_path, "dataset.csv")
# csv_file = open(csv_path, mode='w', newline='', encoding='utf-8')
# csv_writer = csv.writer(csv_file)
# csv_writer.writerow(["label", "img_path", "post_text"])


# # Leer URLs ya guardadas (si existen)
# descargados = set()
# if os.path.exists(csv_path):
#     with open(csv_path, mode='r', encoding='utf-8') as csv_file:
#         reader = csv.DictReader(csv_file)
#         for row in reader:
#             descargados.add(row["img_path"])  # usamos img_path para evitar duplicados


In [ ]:
# with open(csv_path, mode='a', newline='', encoding='utf-8') as csv_file:
#     csv_writer = csv.writer(csv_file)
    
#     # Escribir encabezado solo si el archivo estaba vacío
#     if os.stat(csv_path).st_size == 0:
#         csv_writer.writerow(["label", "img_path", "post_text"])

#     guardadas = 0
#     while guardadas < max_images:
#         posts = driver.find_elements(By.XPATH, '//div[@data-ad-preview="message"]')
#         imagenes = driver.find_elements(By.XPATH, '//img[contains(@src, "scontent")]')

#         for i in range(min(len(posts), len(imagenes))):
#             if guardadas >= max_images:
#                 break

#             texto = posts[i].text.strip()
#             img_url = imagenes[i].get_attribute("src")

#             img_path = os.path.join(img_base_path, f"img_{hash(img_url)}.jpg")  # nombre único

#             if texto and img_url and img_path not in descargados:
#                 try:
#                     img_data = requests.get(img_url).content
#                     with open(img_path, 'wb') as handler:
#                         handler.write(img_data)

#                     texto_limpio = texto.replace("\n", " ")
#                     csv_writer.writerow([class_label, img_path, texto_limpio])

#                     descargados.add(img_path)
#                     guardadas += 1
#                     print(f"[{guardadas}/{max_images}] Imagen guardada: {img_path}")
#                 except Exception as e:
#                     print(f"[X] Error al descargar imagen: {e}")

#         if guardadas < max_images:
#             driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
#             time.sleep(5)

# # Finalizar
# driver.quit()
# print("✅ Dataset actualizado en:", base_path)